# 📓 Cas d'usage IA — [Orientation et tri multimodal des demandeurs d'emploi par l'Intelligence Artificielle]

**Auteur·e** : `Nicolas H`  
**Promo** : ATOS Atlas IA — Parcours 2 (Pros IT)  
**Date début** : `05/09/2026`  
**Dernière mise à jour** : `JJ/MM/AAAA`

---

## 0. Imports & configuration

🎯 Centraliser les imports et la configuration reproductible.  
💡 `random_state=42` partout, `pd.set_option` pour l'affichage si besoin.  


In [1]:
# Imports standards
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Affichage
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

# Imports ML (à activer au fur et à mesure)
# from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline

## 0.5 Traçabilité de la session

🎯 Documenter ce qui rend l'analyse **reproductible** sur tout environnement et par une tierse partie.

💡 Pour une analyse reproductible, trois choses à figer :
- la **version du dataset** (source, date d'extraction, hash si possible),
- les **versions des libs** (cf. `requirements.txt` du repo associé, ou `pip freeze`),
- le **commit Git** du repo associé au notebook.


In [2]:
import sys
import subprocess
from datetime import datetime

# --- Métadonnées du dataset ---
DATASET_NAME = "dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv"          # ex: "UCI Student Performance" ou "FastIA maintenance v1"
DATASET_SOURCE = "Agence Nat. pour l'emploi"        # URL, chemin, ou contact si fourni par le client
DATASET_VERSION = "inconnue"  # date d'extraction OU version explicite

# --- Métadonnées de la session ---
session_date = datetime.now().isoformat(timespec="minutes")
py_version = sys.version.split()[0]

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    git_commit = "non disponible (notebook hors repo Git)"

print(f"Date session : {session_date}")
print(f"Python       : {py_version}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Git commit   : {git_commit}")
print(f"Dataset      : {DATASET_NAME} (source={DATASET_SOURCE}, version={DATASET_VERSION})")

Date session : 2026-09-08T22:31
Python       : 3.14.4
NumPy        : 2.5.2
Pandas       : 3.0.5
Git commit   : 4789392
Dataset      : dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv (source=Agence Nat. pour l'emploi, version=inconnue)


---
## 1. Cadrage métier & problème

🎯 **Cette partie adresse les questions suivantes** :

- Qui est le client ? Quel est son besoin métier réel ?
- Qui sont les bénéficiaires directs et indirects de la solution ?
- Quel type de tâche IA (classification, régression, clustering, génération…) ?
- Quels critères de succès **côté métier** (pas seulement côté modèle) ?
- Quels risques éthiques, réglementaires, de biais sont pressentis ?
- Quel cadre réglementaire s'applique (RGPD, AI Act, secteur régulé) ?


### 1.1 Contexte client

*[Décrire en 5-10 lignes le client, son secteur, le problème métier qu'il rencontre, ce qui motive le recours à l'IA.]*

Le client est un acteur public majeur chargé de l'accompagnement des demandeurs d'emploi et de leur retour à l'emploi. Dans le cadre de ses missions, les conseillers réalisent des entretiens d'orientation afin d'évaluer la situation des usagers et de définir le niveau d'accompagnement le plus adapté. Face à un volume important de dossiers et à la nécessité d'allouer efficacement les ressources d'accompagnement, l'organisme souhaite disposer d'un outil d'aide à la décision capable d'estimer le risque de chômage de longue durée dès le premier entretien. Le projet consiste à exploiter des données administratives, socio-professionnelles, géographiques et textuelles afin d'automatiser partiellement cette évaluation. Le recours à l'intelligence artificielle vise à améliorer la rapidité, l'homogénéité et la fiabilité des décisions d'orientation tout en renforçant la détection précoce des situations nécessitant un accompagnement renforcé

### 1.2 Énoncé du problème IA

*[Reformuler en termes de tâche IA]*

Le problème peut être formulé comme une tâche de classification supervisée multiclasse.
L'objectif est de :
> Prédire la classe de retour à l'emploi d'un demandeur d'emploi à partir d'informations tabulaires (âge, diplôme, ancienneté, statut administratif, données géographiques, etc.) et d'une synthèse textuelle issue du premier entretien.

La variable cible comporte trois catégories :

| Classe | Signification |
|---|---|
| 0 | Retour à l'emploi rapide (< 6 mois) | 
| 1 | Retour à l'emploi moyen (6 à 12 mois) | 
| 2 | Risque de chômage longue durée (> 12 mois) | 

Le système devra attribuer automatiquement l'une de ces trois classes à chaque nouvel usager

### 1.3 Bénéficiaires & impacts

*[Direct : qui utilise la solution ? Indirect : qui subit les conséquences des prédictions ?]*

**Bénéficiaires directs**
- Conseillers de l'agence nationale : Les conseillers utiliseront la solution comme un outil d'aide à la décision lors des entretiens d'orientation. Le modèle fournira une estimation du niveau de risque afin d'aider à identifier rapidement les usagers nécessitant un accompagnement renforcé.

- Managers et responsables des politiques d'emploi : Les responsables opérationnels pourront s'appuyer sur les prédictions produites pour optimiser l'allocation des ressources et le pilotage des dispositifs d'accompagnement.

**Bénéficiaires indirects**
- Demandeurs d'emploi : Les usagers bénéficieront d'une orientation potentiellement plus rapide et plus adaptée à leur situation. Une meilleure détection des situations à risque permettra d'accéder plus tôt à des dispositifs d'accompagnement spécifiques.
- Collectivité publique : Une amélioration de l'efficacité des dispositifs d'accompagnement peut contribuer à réduire les coûts associés au chômage de longue durée et à améliorer les résultats des politiques publiques de l'emploi.

**Impacts négatifs potentiels**
Une mauvaise classification pourrait conduire à une orientation inadaptée. Le risque le plus critique identifié est celui d'un usager relevant réellement de la classe 2 qui serait classé en classe 0, le privant ainsi d'un accompagnement renforcé

### 1.4 Critères de succès

⚠️ Pas d'objectifs chiffrés, a reformuler et renseigner

### 1.5 Risques éthiques & réglementaires anticipés

*[Variables sensibles, biais possibles, RGPD, AI Act, droit sectoriel.]*

**Utilisation de données sensibles**
Le jeu de données contient une variable explicitement identifiée comme sensible : nationalité hors Union Européenne. Son utilisation peut introduire des biais discriminatoires et devra faire l'objet d'une analyse spécifique conformément aux exigences du RGPD et à l'approche éthique demandée dans le sujet.

**Risque de discrimination algorithmique**
Le modèle pourrait apprendre des corrélations historiques défavorables envers certaines catégories de population et reproduire ou amplifier des biais existants. Une comparaison avec et sans variables sensibles devra être réalisée afin de mesurer cet impact.

**Transparence des décisions**
Les prédictions influencent des décisions d'orientation pouvant avoir des conséquences importantes pour les usagers. Il sera donc nécessaire de fournir des éléments d'explicabilité permettant aux conseillers de comprendre les résultats proposés par le modèle.

**Protection des données personnelles**
Les données utilisées concernent des personnes physiques et contiennent des informations socio-professionnelles ainsi que des verbatims d'entretien. Leur traitement devra respecter les principes de minimisation, de sécurité, de traçabilité et de confidentialité prévus par le RGPD.

**Responsabilité juridique**
Une erreur de prédiction peut conduire à une perte de chance pour un usager si celui-ci ne bénéficie pas d'un accompagnement adapté. La responsabilité de l'administration et du concepteur du système doit donc être prise en compte.


### 1.6 📝 Synthèse cadrage

Le projet vise à développer un système d'intelligence artificielle capable de prédire le délai probable de retour à l'emploi d'un demandeur d'emploi à partir de données administratives, socio-professionnelles, géographiques et textuelles. La problématique est modélisée sous la forme d'une classification supervisée à trois classes : retour rapide, retour moyen ou risque de chômage longue durée. Le principal critère de succès consiste à réduire les erreurs critiques conduisant à sous-estimer le niveau de risque d'un usager en difficulté. Le risque éthique majeur identifié concerne l'utilisation de variables sensibles susceptibles d'introduire des biais discriminatoires dans les décisions d'orientation.


---
## 2. Identification & acquisition des données

🎯 Décrire **les données à disposition**, avant tout traitement.


### 2.1 Sources

| Source | Type | Volumétrie | Format | Accès | Date extraction |
|---|---|---|---|---|---|
| Agence nat. pour l'emploi | CSV | 2 500 | CSV, sép "*virgule*" | copié dans /data | *inconnue* |


In [3]:
# 2.2 Chargement
DATA_DIR = Path("../data")
df = pd.read_csv(DATA_DIR / "dataset_trajectoire_emploi.csv")

print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")

df.columns.tolist()

Nombre de lignes : 2500
Nombre de colonnes : 10


['usager_id',
 'age',
 'niveau_diplome',
 'anciennete_poste_ans',
 'code_rome_vise',
 'code_insee_commune',
 'est_allocataire',
 'nationalite_hors_ue',
 'synthese_entretien',
 'classe_retour_emploi']

### 2.3 Dictionnaire de variables

| Variable | Description | Type | Plage / valeurs | Cible ? | Sensible ? |
|---|---|---|---|---|---|
| `usager_id` | *Identifiant unique du demandeur d'emploi* | num/cat | *[ID_0000]* | ❌ | ❌ |
| `age` | *Âge de l'usager en années* | num | *[> 18]* | ❌ | ❌/✅ |
| `niveau_diplome` | *Plus haut niveau de diplôme obtenu* | cat | *Sans diplôme, Bac, Bac+2, Bac+5* | ❌ | ❌/✅ |
| `anciennete_poste_ans` | *Nombre d'années d'expérience dans le dernier emploi occupé* | num | *[min-max ou modalités]* | ❌ | ❌/✅ |
| `code_rome_vise` | *Code du Répertoire Opérationnel des Métiers et des Emplois ciblé par l'usager* | cat | *[5 caractères]* | ❌ | ❌/✅ |
| `code_insee_commune` | *Code officiel géographique INSEE de la commune de résidence * | cat | *[5 digits]* | ❌ | ❌/✅ |
| `est_allocataire` | *Statut d'indemnisation de l'usager* | cat | *[1 (Oui), 0 (Non)]* | ❌ | ❌/✅ |
| `nationalite_hors_ue` | *Variable sensible : * | cat | *[1 (hors Union Européenne), 0 (UE)]* | ❌ | ✅ |
| `synthèse_entretien` | *Texte libre contenant les notes textuelles du conseiller lors de l'entretien * | texte | *[texte libre]* | ❌ | ❌ |
| `classe_retour_emploi` | Variable cible : 0 (Rapide < 6m), 1 (Moyen 6-12m), 2 (Risque de longue durée > 12m) | cat | *[0,1,2]* | ✅ | ❌ |

### 2.4 📝 Synthèse acquisition

Le jeu de données contient des variables numériques, catégorielles et textuelles permettant d'alimenter un modèle de classification multiclasse.

La présence d'une variable sensible ("nationalité hors UE") nécessitera une analyse éthique spécifique. Cette variable peut en effet être source de biais ou de discrimination indirecte. Un entrainement sans cette variable pourra faire l'objet d'un scénario alternatif.

Les prochaines étapes consisteront à réaliser l'analyse exploratoire approfondie (EDA), à étudier la qualité des données et à identifier les variables les plus informatives pour la prédiction.




---
## 3. Exploration & analyse des données (EDA)

> 🎯 Comprendre la donnée **avant** de la transformer. Détecter manquants, doublons, aberrations, déséquilibres, biais.

L'analyse exploratoire est conduite selon une démarche progressive consistant à évaluer la qualité du jeu de données, à caractériser les variables individuellement puis à étudier leurs interactions afin de mieux comprendre les mécanismes associés au retour à l'emploi et d'orienter les étapes ultérieures de préparation et de modélisation. L'objectif de cette analyse exploratoire est de répondre aux questions suivantes :

	• Les données sont-elles complètes et exploitables ?
	• La variable cible présente-t-elle un déséquilibre susceptible d'impacter l'apprentissage ?
	• Quelles sont les caractéristiques des demandeurs d'emploi selon leur classe de retour à l'emploi ?
	• Existe-t-il des variables particulièrement discriminantes pour la prédiction ?
	• Certaines variables présentent-elles des risques de biais ou de fuite d'information ?

Pour répondre à ces questions, plusieurs types de visualisations seront mobilisés :

	• Diagrammes en barres pour analyser la répartition des modalités et de la variable cible ;
	• Histogrammes pour étudier les distributions des variables numériques ;
	• Boxplots pour comparer les distributions selon les classes de la cible ;
	• Matrices de corrélation pour identifier d'éventuelles relations entre variables quantitatives ;
	• Visualisations spécifiques aux variables textuelles afin d'évaluer leur richesse informationnelle.

Les enseignements issus de cette phase permettront de définir la stratégie de préparation des données, de sélectionner les variables pertinentes et d'orienter les choix de modélisation tout en tenant compte des contraintes métier et des enjeux de performance du projet.


---
## 4. Préparation des données

🎯 Construire un **pipeline reproductible** de préprocessing (pas de transformations one-shot dispersées).


### Scénarios de jeux de données

*[Pour la certif, anticipe : 2 à 4 scénarios = avec/sans variables sensibles, avec/sans certaines variables corrélées à la cible. Liste-les ici, tu les compareras en section 6.]*

| Scénario | Variables conservées | Justification |
|---|---|---|
| S1 | Toutes | Baseline |
| S2 | Sans variables sensibles | Atténuation biais |
| S3 | … | … |

### 📝 Synthèse préparation

*[Choix de pipeline, scénarios construits, traçabilité.]*

---
## 5. Modélisation & entraînement

🎯 **Comparer plusieurs modèles** avec validation croisée, choisir, justifier.



### 5.1 Modèles candidats



---
## 6. Analyse des scénarios & arbitrages

🎯 **Comparer les scénarios définis en 4.3** et argumenter le compromis performance / éthique / robustesse.


## 6. Analyse des scénarios & arbitrages

### 6.1 Tableau comparatif



### 6.2 Recommandation finale


### 6.3 📝 Synthèse arbitrages



---
## Industrialisation

**Compétences** : C6 (implémenter le modèle), C7 (architecture cible)

🎯 Transformer le modèle en **service exploitable** : API, UI, conteneurs, tracking d'expériences.

> ⚠️ **Cette section ne contient PAS le code de l'API/UI.** Le code vit dans le **repo Git** associé.  
> Dans le notebook, on met uniquement :
> 1. **Description architecturale** (texte)
> 2. **1 schéma** d'architecture (Mermaid, draw.io export, ou image)
> 3. **Lien vers le repo Git** + dossiers concernés
> 4. **Captures d'écran clés** (UI, dashboard, OpenAPI Swagger, workflow CI/CD vert)
> 5. **Justifications des choix techniques** (pourquoi FastAPI plutôt que Flask, pourquoi MLflow plutôt que `experiments.md`, etc.)

⚠️ Attendus certif (présents dans le repo, **référencés** dans cette section du notebook) :  
- API FastAPI avec routes `/predict`, `/train`, `/health`  
- Validation Pydantic des entrées  
- Logging structuré (Loguru)  
- Tests pytest  
- Conteneurisation Docker  
- CI/CD GitHub Actions  
- Tracking MLflow

---
## Suivi en production & amélioration continue

🎯 Anticiper la dérive et la maintenance du modèle après mise en prod.


---
## 📎 Annexes

### A. Glossaire métier & technique

### B. Sources & bibliographie

- *[Datasets, articles, documentation technique consultés]*

### C. Décisions techniques détaillées

*[Justifications longues qu'on ne met pas dans le corps du notebook pour le garder lisible.]*

### D. Journal de bord

